# Модуль анализа и прогнозирования продаж малого бизнеса

Демонстрация работы программы на одном файле - путь глазами пользователя:

1. **Исходные данные** - файл как есть
2. **Подготовка** - сопоставление колонок, очистка, единая модель данных
3. **Аналитика** - динамика, последний месяц, топ товаров, детально по товару
4. **Прогноз** - автоматический выбор модели и горизонта, отчёт о точности
5. **Точность** - сравнение моделей и подбор горизонта
6. **Сводная таблица** по трём тестовым бизнесам

Графики сохраняются в `notebooks/figures/` - их можно вставлять в текст курсовой.

In [ ]:
# Подготовка: чтобы ноутбук видел папку app
import sys

sys.path.append("..")

In [ ]:
import json
from pathlib import Path

from matplotlib.figure import Figure

from app.analytics import (
    breakdown,
    by_period,
    incomplete_periods,
    item_detail,
    last_full_month_vs_previous,
)
from app.charts import bar_chart, forecast_chart, line_chart
from app.config import REAL_DIR, SYNTHETIC_DIR  # noqa: F401 - REAL_DIR для примера ниже
from app.data import load, prepare_events, suggest_exclusions
from app.forecast import (
    STEP_FORECAST,
    compare_models,
    horizon_scan,
    make_forecast,
    models_for,
    to_daily,
)

## Параметры

Чтобы проверить программу на своём файле: положи его в `data/real/`, поменяй `FILE`, `MAPPING` и `COUNTRY` и запусти все ячейки (Run → Run All Cells).

Роли колонок: `date` - дата, `object` - товар или услуга, `category` - группа, `quantity` - количество, `amount` - сумма, `price` - цена за единицу (если суммы нет, она посчитается как цена × количество), `receipt` - номер чека, `executor` - сотрудник, `attribute` - другой текстовый признак (артикул, бренд, страна), `measure` - другое число (скидка), `person` - клиент (в анализ не попадёт), `ignore` - не нужна.

Пример для настоящего набора Online Retail (UCI) - см. ячейку ниже, закомментированный вариант.

In [ ]:
FILE = SYNTHETIC_DIR / "shop.xlsx"
COUNTRY = "BY"  # чьи праздники учитывать в прогнозе: BY, RU, GB...
CURRENCY = "р."  # валюта для графиков и текстов: "р.", "$", "£"...
EXCLUDE = []  # служебные строки, которые не учитывать (доставка, комиссии) - см. подсказку ниже

# Какая колонка файла что значит - в курсовой это указывает пользователь
MAPPING = {
    "Дата": "date",
    "Артикул": "attribute",
    "Номенклатура": "object",
    "Бренд": "attribute",
    "Группа": "category",
    "Количество": "quantity",
    "Сумма": "amount",
}

# Настоящий интернет-магазин из Великобритании (UCI Online Retail) - раскомментируй:
# FILE = REAL_DIR / "Online Retail.xlsx"
# COUNTRY = "GB"
# CURRENCY = "£"
# EXCLUDE = ["DOTCOM POSTAGE", "POSTAGE", "CARRIAGE", "Next Day Carriage", "Manual",
#            "AMAZON FEE", "Bank Charges", "CRUK Commission", "Discount", "Adjust bad debt"]
# MAPPING = {
#     "InvoiceNo": "receipt",
#     "StockCode": "attribute",
#     "Description": "object",
#     "Quantity": "quantity",
#     "InvoiceDate": "date",
#     "UnitPrice": "price",
#     "CustomerID": "person",
#     "Country": "attribute",
# }

FIGURES_DIR = Path("figures")  # сюда сохраняются графики для курсовой
FIGURES_DIR.mkdir(exist_ok=True)


def save(figure: Figure, name: str) -> Figure:
    # Сохраняет график в PNG (для курсовой) и возвращает его, чтобы он показался в ноутбуке
    figure.savefig(FIGURES_DIR / f"{name}.png", dpi=200)
    return figure


def show(report: dict) -> None:
    # Печатает отчёт построчно: "ключ: значение"
    for key, value in report.items():
        print(f"{key}: {value}")

## 1. Исходные данные

Файл в том виде, в каком его выгрузил бизнес: шапка отчёта сверху, суммы, строка «Итого» в конце.

In [ ]:
sheets = load(FILE)
raw = next(iter(sheets.values()))  # первый лист
print(f"Файл: {FILE.name} | листов: {len(sheets)} | строк: {len(raw)}")
print("Колонки:", list(raw.columns))
raw.head()

In [ ]:
raw.tail(3)  # в конце - итоговая строка, её программа уберёт

## 2. Подготовка данных

Загрузка → очистка (даты, числа, строки «Итого») → единая модель данных. Одна команда.

In [ ]:
events, report = prepare_events(FILE, MAPPING, exclude=EXCLUDE)
show(report)

# Подсказка: что ещё похоже на служебные строки. Решает человек - среди предложенных
# бывают и настоящие товары (например, "CARRIAGE CLOCK" - часы, а не доставка)
suggested = suggest_exclusions(events)
if suggested:
    print("\nПохоже на служебные строки (можно добавить в EXCLUDE):", suggested)

In [ ]:
# Единая модель: одинаковые колонки для любого бизнеса + дополнительные поля в конце
events.head()

## 3. Аналитика

In [ ]:
# Динамика выручки по месяцам - только полные месяцы: неполный выглядел бы как обвал
skipped = incomplete_periods(events)
if skipped:
    print("Неполные месяцы не показаны:", ", ".join(skipped))
monthly = by_period(events, "месяц", full_periods_only=True)
save(line_chart(monthly, "Выручка по месяцам", CURRENCY), "1_выручка_по_месяцам")

In [ ]:
# Как прошёл последний полный месяц по сравнению с предыдущим
show(last_full_month_vs_previous(events))

In [ ]:
# Что продаётся лучше всего
top = breakdown(events)
save(bar_chart(top, "Топ товаров по выручке", CURRENCY), "2_топ_товаров")

In [ ]:
# Детально по лидеру продаж
leader = item_detail(events, top.index[0])
show({key: value for key, value in leader.items() if key != "динамика"})
leader_title = f"{leader['товар']}: выручка по месяцам"
save(line_chart(leader["динамика"], leader_title, CURRENCY), "3_лидер")

## 4. Прогноз

Программа сама определяет ступеньку холодного старта, проводит экзамен моделей на прошлом, выбирает лучшую модель и горизонт, строит прогноз и объясняет, насколько ему можно верить.

In [ ]:
result = make_forecast(events, country=COUNTRY, currency=CURRENCY)
print("Ступенька:", result["ступенька"])
print(result["сообщение"])
if result["ступенька"] == STEP_FORECAST:
    print("Модель:", result["модель"], "| горизонт:", result["горизонт_дней"], "дн.")
    print("Причины:")
    for reason in result["точность"]["причины"]:
        print("  -", reason)

In [ ]:
if result["ступенька"] == STEP_FORECAST:
    title = f"Прогноз выручки на {result['горизонт_дней']} дн. ({result['модель']})"
    chart = forecast_chart(to_daily(events), result["прогноз"], title, currency=CURRENCY)
    display(save(chart, "4_прогноз"))

## 5. Точность прогноза

Экзамен на прошлом: прячем конец истории, прогнозируем его по тому, что было раньше, сравниваем с правдой. Три отрезка, среднее.

- **MAE** - средняя ошибка за день, р.
- **WAPE_%** - ошибка в % от продаж
- **ошибка_итога_%** - насколько ошибся в сумме за период (+ завысил, - занизил)
- **в_коридоре_%** - сколько дней попало в коридор (обещано 80%)

In [ ]:
daily = to_daily(events)
models = models_for(COUNTRY)  # модели с праздниками нужной страны
compare_models(daily, models=models)  # на горизонте 30 дней

In [ ]:
# Ошибка Prophet на разных горизонтах
horizon_scan(daily, models["Prophet"])

## 6. Сводная таблица по трём бизнесам

Для главы «Тестирование»: одна и та же программа на трёх разных бизнесах.

In [ ]:
import pandas as pd

rows = []
for file_name, business in [
    ("shop.xlsx", "магазин"),
    ("cafe.csv", "кафе"),
    ("salon.xlsx", "салон"),
]:
    path = SYNTHETIC_DIR / file_name
    mapping = json.loads(path.with_suffix(".json").read_text(encoding="utf-8"))
    business_events, _ = prepare_events(path, mapping)
    comparison = compare_models(to_daily(business_events))
    forecast = make_forecast(business_events)
    rows.append(
        {
            "бизнес": business,
            "история, дн.": forecast["история_дней"],
            "Prophet, ошибка %": comparison.loc["Prophet", "WAPE_%"],
            "простой прогноз, ошибка %": comparison.loc["простой прогноз", "WAPE_%"],
            "Prophet, ошибка итога %": comparison.loc["Prophet", "ошибка_итога_%"],
            "Prophet, в коридоре %": comparison.loc["Prophet", "в_коридоре_%"],
            "выбранная модель": forecast.get("модель", "-"),
            "горизонт, дн.": forecast.get("горизонт_дней", "-"),
            "точность": forecast["точность"]["точность"] if "точность" in forecast else "-",
        }
    )

summary = pd.DataFrame(rows).set_index("бизнес")
summary